# שיעור 21 — קנה מידה ותקנון לקראת KNN
יחידה 3 | 2026–2027 | StandardScaler מאימון בלבד.

ידע קודם: שונות וסטיית תקן, X/y, חלוקה והשלמה משיעור 20. התאמה שלנו לסיכום מפגש 7 בהשתלמות, לא מחברת מקור של המרצה.
[CSV](https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv) | [מקור Palmer Penguins](https://github.com/allisonhorst/palmerpenguins) | [קטלוג](https://github.com/samiabo75/Data-Science-ML-Course/blob/main/Datasets/README.md).
ייחוס: Horst, Hill & Gorman (2020), palmerpenguins, DOI 10.5281/zenodo.3960218; גרסת הקורס 344×7. פרטי שימוש במקור. שמרו עותק אישי.

## למה לומדים את זה עכשיו?
להשוות קילומטרים לשעות בלי קנה מידה מתאים עלול לשנות מי נחשב קרוב.
**בתהליך הפרויקט:** הכנה.
**מסגרת פעילה:** 0.5 שעות לימוד — 22.5 דקות, כולל דוגמה ותרגול עצמאי. המורה בוחר את משימות הליבה שבמדריך; שאר הפעילויות לחזרה או לעיון, ללא תוספת שעות.

## פתיחה — האם היחידה יכולה לשנות מי קרוב יותר?
ב־KNN נבחר בעתיד שכנים לפי מרחק בין מאפיינים. נמחיש בעיה בדוגמה מומצאת: תצפית חדשה היא אורך מקור 40 מ״מ ומסה 4000 גרם. דוגמה A היא (41,4500), ודוגמה B היא (45,4100).

סכום ריבועי ההפרשים מהתצפית החדשה, ביחידות הגולמיות: A נותנת 1²+500²=250001; B נותנת 5²+100²=10025, ולכן B קרובה יותר לפי חישוב זה. אם מבטאים מסה בק״ג, אותו חישוב נותן A:1²+0.5²=1.25; B:5²+0.1²=25.01 — כעת A קרובה יותר.

הישויות לא השתנו; יחידת המדידה שינתה את משקל המאפיין בחישוב. זו המחשה מדוע צריך לבחור קנה מידה במודע. אין כאן טענה שלחיבור יחידות גולמיות שונות יש משמעות פיזיקלית טבעית, או שהקרוב יותר מוכרח לשפר חיזוי.

## 1. תקנון — פירוש לפני כלי
לכל מאפיין בנפרד: z=(x−ממוצע האימון)/סטיית התקן של האימון. אם ממוצע אורך באימון 40 מ״מ וסטיית תקן 2 מ״מ, אורך 44 מקבל (44−40)/2=2. פירוש: שתי סטיות תקן מעל ממוצע האימון, ללא יחידת מ״מ.

StandardScaler משתמש בסטיית תקן עם ddof=0. הוא אינו הופך התפלגות לנורמלית, אינו מסיר חריגים ואינו מבטיח שיפור דיוק. הוא גם רגיש לערכי קצה.

אם מאפיין קבוע באימון, הספרייה משתמשת בגורם קנה מידה 1 למאפיין זה; אין לבצע ידנית חלוקה ב־0.

## 2. מכינים נתונים כמו בשיעור 20
התא הבא הוא תשתית מוכנה לחזרה. המאפיינים הפעם הם אורך סנפיר במ״מ ומסה בגרם. לא לומדים פרמטרים לפני החלוקה.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
model_df = df.dropna(subset=["species"]).copy()
X = model_df[["flipper_length_mm", "body_mass_g"]]
y = model_df["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)
print("אימון:", X_train_ready.shape, "בדיקה:", X_test_ready.shape)

## 3. fit — לומדים את סרגל האימון
StandardScaler יוצר כלי. set_output מבקשת שהתוצאות יישארו טבלאות Pandas, לנוחות קריאה ושמירת שמות ואינדקסים. fit לומדת את הממוצעים וקני המידה מ־X_train_ready, אחרי ההשלמה.

In [ ]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
scaler.fit(X_train_ready)
print("ממוצעי האימון שנלמדו:", scaler.mean_)
print("קני המידה שנלמדו:", scaler.scale_)

mean_ ו־scale_ מציגים ערכים לפי סדר העמודות שנבחר. אלה פרמטרים שנלמדו מנתונים, לא תוצאות חיזוי. אין fit על X_test_ready.

## 4. transform — משתמשים באותו סרגל לשתי הקבוצות

In [ ]:
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head(3))
display(X_test_scaled.head(3))
print("ממוצעי אימון לאחר תקנון:")
print(X_train_scaled.mean())
print("סטיות תקן אימון, ddof=0:")
print(X_train_scaled.std(ddof=0))

במאפיינים הלא־קבועים כאן ממוצעי האימון קרובים ל־0 וסטיות התקן ל־1, עד שגיאת ייצוג מספרי. אין לדרוש שממוצעי הבדיקה יהיו 0 או סטיות התקן שלה 1 — היא מומרת לפי פרמטרי האימון.

ערך שלילי אינו שגיאה: הוא מתחת לממוצע האימון. סדר הרשומות ושמות המאפיינים נשמרו; y לא תוקננה.

## 5. הרחבת רשות — בדיקה רחוקה מהאימון
לנתוני הוראה מומצאים: אימון 10,20,30; בדיקה 1000. נלמד סרגל אחד מן האימון; בדיקה רחוקה יכולה לקבל ערך מתוקנן גדול מאוד. התקנון אינו מגביל ערכים ל־0–1.

In [ ]:
toy_train = pd.DataFrame({"value": [10, 20, 30]})
toy_test = pd.DataFrame({"value": [1000]})
toy_scaler = StandardScaler()
toy_scaler.set_output(transform="pandas")
toy_scaler.fit(toy_train)
print("בדיקה לפי סרגל האימון:")
print(toy_scaler.transform(toy_test))

## תוצר לפרויקט — יומן תקנון
תעדו מדוע נחוץ קנה מידה למודל המתוכנן, שמות ויחידות מקוריות, סדר הכנה, קבוצת fit, פרמטרים, ממדים ופירוש התוצאה. אין לבחור Scaler לפי ניסויים חוזרים על Test. באימות צולב עתידי גם ההשלמה והתקנון ילמדו מחדש בתוך כל קפל אימון.

## כרטיס יציאה
1. מה fit לומדת ומה transform עושה?
2. מאילו נתונים מותר ללמוד סרגל?
3. האם Test חייבת לקבל ממוצע 0?
4. האם תקנון מוכיח נורמליות או דיוק גבוה יותר?
5. למה יחידות שונות יכולות לשנות חישוב מרחק?

אין אימון KNN בשיעור זה. בהמשך נשתמש בהכנה בתוך תהליך מודל מסודר. Penguins להוראה, לא מאגר גמר מאושר.

In [ ]:
print("מקור:", df.shape)
print("אימון מתוקנן:", X_train_scaled.shape)
print("בדיקה מתוקננת:", X_test_scaled.shape)